In [2]:
import pandas as pd
import numpy as np
import re

In [7]:
dataset = pd.read_csv('./dataset/train.csv')
print(dataset.head())
print(dataset.columns)
print(dataset.info())

   sample_id                                    catalog_content  \
0      33127  Item Name: La Victoria Green Taco Sauce Mild, ...   
1     198967  Item Name: Salerno Cookies, The Original Butte...   
2     261251  Item Name: Bear Creek Hearty Soup Bowl, Creamy...   
3      55858  Item Name: Judee’s Blue Cheese Powder 11.25 oz...   
4     292686  Item Name: kedem Sherry Cooking Wine, 12.7 Oun...   

                                          image_link  price  
0  https://m.media-amazon.com/images/I/51mo8htwTH...   4.89  
1  https://m.media-amazon.com/images/I/71YtriIHAA...  13.12  
2  https://m.media-amazon.com/images/I/51+PFEe-w-...   1.97  
3  https://m.media-amazon.com/images/I/41mu0HAToD...  30.34  
4  https://m.media-amazon.com/images/I/41sA037+Qv...  66.49  
Index(['sample_id', 'catalog_content', 'image_link', 'price'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 4 columns):
 #   Column           Non-Null Count  

### Remove all kinds of Emoji's and Weird Characters 

In [8]:
emoji_pattern = re.compile(
    '['
    '\U0001F600-\U0001F64F'
    '\U0001F300-\U0001F5FF'
    '\U0001F680-\U0001F6FF'
    '\U0001F700-\U0001F77F'
    '\U0001F780-\U0001F7FF'
    '\U0001F800-\U0001F8FF'
    '\U0001F900-\U0001F9FF'
    '\U0001FA00-\U0001FA6F'
    '\U0001FA70-\U0001FAFF'
    '\U00002702-\U000027B0'
    '\U000024C2-\U0001F251'
    ']+',
    flags=re.UNICODE
)

specific_chars_pattern = r'[–—’\U0001F600-\U0001F64F]'

In [9]:
dataset['cleaned_column'] = dataset['catalog_content'].str.replace(emoji_pattern, '', regex=True)
dataset['cleaned_column'] = dataset['cleaned_column'].str.replace(specific_chars_pattern, ',', regex=True)
dataset

,sample_id,catalog_content,image_link,price,cleaned_column
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.890,"Item Name: La Victoria Green Taco Sauce Mild, ..."
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.120,"Item Name: Salerno Cookies, The Original Butte..."
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.970,"Item Name: Bear Creek Hearty Soup Bowl, Creamy..."
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.340,"Item Name: Judee,s Blue Cheese Powder 11.25 oz..."
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.490,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun..."
...,...,...,...,...,...
74995,41424,Item Name: ICE BREAKERS Spearmint Sugar Free M...,https://m.media-amazon.com/images/I/81p9PcPsff...,10.395,Item Name: ICE BREAKERS Spearmint Sugar Free M...
74996,35537,"Item Name: Davidson's Organics, Vanilla Essenc...",https://m.media-amazon.com/images/I/51DDKoa+mb...,35.920,"Item Name: Davidson's Organics, Vanilla Essenc..."
74997,249971,Item Name: Jolly Rancher Hard Candy - Blue Ras...,https://m.media-amazon.com/images/I/91R2XCcpUf...,50.330,Item Name: Jolly Rancher Hard Candy - Blue Ras...
74998,188322,Item Name: Nescafe Dolce Gusto Capsules - CARA...,https://m.media-amazon.com/images/I/51W40YU98+...,15.275,Item Name: Nescafe Dolce Gusto Capsules - CARA...


In [10]:
print("Second one:")
print(repr(dataset['catalog_content'].iloc[1]))
print(repr(dataset['cleaned_column'].iloc[1]))

Second one:
'Item Name: Salerno Cookies, The Original Butter Cookies, 8 Ounce (Pack of 4)\nBullet Point 1: Original Butter Cookies: Classic butter cookies made with real butter\nBullet Point 2: Variety Pack: Includes 4 boxes with 32 cookies total\nBullet Point 3: Occasion Perfect: Delicious cookies for birthdays, weddings, anniversaries\nBullet Point 4: Shareable Treats: Fun to give and enjoy with friends and family\nBullet Point 5: Salerno Brand: Trusted brand of delicious butter cookies since 1925\nValue: 32.0\nUnit: Ounce\n'
'Item Name: Salerno Cookies, The Original Butter Cookies, 8 Ounce (Pack of 4)\nBullet Point 1: Original Butter Cookies: Classic butter cookies made with real butter\nBullet Point 2: Variety Pack: Includes 4 boxes with 32 cookies total\nBullet Point 3: Occasion Perfect: Delicious cookies for birthdays, weddings, anniversaries\nBullet Point 4: Shareable Treats: Fun to give and enjoy with friends and family\nBullet Point 5: Salerno Brand: Trusted brand of delicious

In [14]:
def parse_catalog(content):
    lines = content.split('\n')
    data = {}
    bullet_points = []
    for line in lines:
        if ': ' in line:
            label, value = line.split(': ', 1)
            if label.startswith('Bullet Point'):
                bullet_points.append(value)
            elif label == 'Item Name':
                data['Item Name'] = value
            elif label == 'Product Description':
                data['Product Description'] = value
            elif label == 'Value':
                data['Value'] = value
            elif label == 'Unit':
                data['Unit'] = value
    data['Bullet Points'] = '; '.join(bullet_points) if bullet_points else None
    return data

In [15]:
def parse_catalog_improved(content):
    """
    Parse catalog content and extract information into structured format.
    Bullet points are stored as arrays, missing fields are set to NaN.
    """
    if pd.isna(content) or content == '':
        return {
            'Item Name': np.nan,
            'Product Description': np.nan,
            'Value': np.nan,
            'Unit': np.nan,
            'Bullet Points': np.nan
        }
    
    lines = content.split('\n')
    data = {
        'Item Name': np.nan,
        'Product Description': np.nan,
        'Value': np.nan,
        'Unit': np.nan,
        'Bullet Points': np.nan
    }
    bullet_points = []
    
    for line in lines:
        line = line.strip()
        if ': ' in line:
            label, value = line.split(': ', 1)
            label = label.strip()
            value = value.strip()
            
            if label.startswith('Bullet Point'):
                if value:  # Only add non-empty bullet points
                    bullet_points.append(value)
            elif label == 'Item Name':
                data['Item Name'] = value if value else np.nan
            elif label == 'Product Description':
                data['Product Description'] = value if value else np.nan
            elif label == 'Value':
                data['Value'] = value if value else np.nan
            elif label == 'Unit':
                data['Unit'] = value if value else np.nan
    
    # Store bullet points as array if any exist, otherwise NaN
    data['Bullet Points'] = bullet_points if bullet_points else np.nan
    
    return data

In [16]:
# Apply the parsing function to the cleaned_column (or catalog_content if cleaned_column doesn't exist)
# Use cleaned_column if it exists, otherwise use catalog_content
content_column = 'cleaned_column' if 'cleaned_column' in dataset.columns else 'catalog_content'

print(f"Parsing content from column: {content_column}")
print(f"Dataset shape before parsing: {dataset.shape}")

# Parse each row and create a list of dictionaries
parsed_data = []
for idx, content in enumerate(dataset[content_column]):
    if idx % 1000 == 0:  # Progress indicator
        print(f"Processing row {idx}...")
    parsed_data.append(parse_catalog_improved(content))

# Convert to DataFrame
parsed_df = pd.DataFrame(parsed_data)

# Add the new columns to the original dataset
for col in ['Item Name', 'Product Description', 'Value', 'Unit', 'Bullet Points']:
    dataset[col] = parsed_df[col]

print(f"Dataset shape after parsing: {dataset.shape}")
print(f"New columns added: {['Item Name', 'Product Description', 'Value', 'Unit', 'Bullet Points']}")

Parsing content from column: cleaned_column
Dataset shape before parsing: (75000, 10)
Processing row 0...
Processing row 1000...
Processing row 2000...
Processing row 3000...
Processing row 4000...
Processing row 5000...
Processing row 6000...
Processing row 7000...
Processing row 8000...
Processing row 9000...
Processing row 10000...
Processing row 11000...
Processing row 12000...
Processing row 13000...
Processing row 14000...
Processing row 15000...
Processing row 16000...
Processing row 17000...
Processing row 18000...
Processing row 19000...
Processing row 20000...
Processing row 21000...
Processing row 22000...
Processing row 23000...
Processing row 24000...
Processing row 25000...
Processing row 26000...
Processing row 27000...
Processing row 28000...
Processing row 29000...
Processing row 30000...
Processing row 31000...
Processing row 32000...
Processing row 33000...
Processing row 34000...
Processing row 35000...
Processing row 36000...
Processing row 37000...
Processing row 

In [17]:
# Display sample of the parsed data
print("Sample of parsed data:")
print("="*50)
sample_cols = ['Item Name', 'Product Description', 'Value', 'Unit', 'Bullet Points']
print(dataset[sample_cols].head())

print("\n" + "="*50)
print("Data types of new columns:")
for col in sample_cols:
    print(f"{col}: {dataset[col].dtype}")

print("\n" + "="*50)
print("Sum of null/NaN values in each column:")
null_counts = {}
for col in sample_cols:
    # Count NaN values
    null_count = dataset[col].isna().sum()
    null_counts[col] = null_count
    print(f"{col}: {null_count}")

print(f"\nTotal rows in dataset: {len(dataset)}")
print(f"Percentage of missing values:")
for col, count in null_counts.items():
    percentage = (count / len(dataset)) * 100
    print(f"{col}: {percentage:.2f}%")

Sample of parsed data:
                                           Item Name  \
0  La Victoria Green Taco Sauce Mild, 12 Ounce (P...   
1  Salerno Cookies, The Original Butter Cookies, ...   
2  Bear Creek Hearty Soup Bowl, Creamy Chicken wi...   
3  Judee,s Blue Cheese Powder 11.25 oz - Gluten-F...   
4  kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...   

                                 Product Description  Value   Unit  \
0                                                NaN   72.0  Fl Oz   
1                                                NaN   32.0  Ounce   
2                                                NaN   11.4  Ounce   
3  Judees Powdered Blue Cheese cheddar cheese pow...  11.25  Ounce   
4                                                NaN   12.0  Count   

                                       Bullet Points  
0                                                NaN  
1  [Original Butter Cookies: Classic butter cooki...  
2  [Loaded with hearty long grain wild rice and v...  

In [18]:
# Show examples of bullet points arrays
print("Examples of Bullet Points arrays:")
print("="*50)

# Find rows with bullet points (not NaN)
rows_with_bullets = dataset[dataset['Bullet Points'].notna()]
print(f"Rows with bullet points: {len(rows_with_bullets)}")

# Show first few examples
for i, (idx, row) in enumerate(rows_with_bullets.head(3).iterrows()):
    print(f"\nExample {i+1} (Row {idx}):")
    print(f"Item Name: {row['Item Name']}")
    print(f"Bullet Points ({len(row['Bullet Points'])} items):")
    for j, bullet in enumerate(row['Bullet Points']):
        print(f"  {j+1}. {bullet}")

# Show statistics about bullet points
bullet_lengths = rows_with_bullets['Bullet Points'].apply(len)
print(f"\nBullet Points Statistics:")
print(f"Average number of bullet points per item: {bullet_lengths.mean():.2f}")
print(f"Max number of bullet points: {bullet_lengths.max()}")
print(f"Min number of bullet points: {bullet_lengths.min()}")

# Verify that cleaned columns were created correctly
print(f"\nFinal dataset shape: {dataset.shape}")
print(f"Final dataset columns: {list(dataset.columns)}")

Examples of Bullet Points arrays:
Rows with bullet points: 60723

Example 1 (Row 1):
Item Name: Salerno Cookies, The Original Butter Cookies, 8 Ounce (Pack of 4)
Bullet Points (5 items):
  1. Original Butter Cookies: Classic butter cookies made with real butter
  2. Variety Pack: Includes 4 boxes with 32 cookies total
  3. Occasion Perfect: Delicious cookies for birthdays, weddings, anniversaries
  4. Shareable Treats: Fun to give and enjoy with friends and family
  5. Salerno Brand: Trusted brand of delicious butter cookies since 1925

Example 2 (Row 2):
Item Name: Bear Creek Hearty Soup Bowl, Creamy Chicken with Rice, 1.9 Ounce (Pack of 6)
Bullet Points (5 items):
  1. Loaded with hearty long grain wild rice and vegetables
  2. Full of hearty goodness
  3. Single serve bowls
  4. Easy to prepare mix
  5. 0 grams trans fat

Example 3 (Row 3):
Item Name: Judee,s Blue Cheese Powder 11.25 oz - Gluten-Free and Nut-Free - Use in Seasonings and Salad Dressings - Great for Dips, Spreads and 

In [24]:
dataset = dataset.drop(columns=['catalog_content', 'cleaned_column'])

In [25]:
len(dataset.columns)

8

### Finally Save the Cleaned Dataset

In [26]:
dataset.to_csv('./dataset/cleaned_train.csv', index=False)

In [28]:
for col in dataset.columns:
    print(f"{col}: {dataset[col].dtype}, Nulls: {dataset[col].isna().sum()}")

sample_id: int64, Nulls: 0
image_link: object, Nulls: 0
price: float64, Nulls: 0
Item Name: object, Nulls: 8
Bullet Points: object, Nulls: 14277
Product Description: object, Nulls: 42475
Value: object, Nulls: 0
Unit: object, Nulls: 1
